# Extracting the dictionary from annotated files

In [ ]:
import xml.etree.ElementTree as ET
import json
import csv

In [ ]:
import xml.etree.ElementTree as ET
import json
from collections import defaultdict

def extract_word_morph_gloss_dict(
    filename,
    word_tiername,
    morph_tiername,
    gloss_tiername,
    existing_dict=None,
    save_json_path="word_morph_gloss_dict.json",
    warnings_counter=None
):
    """
    Build / extend dictionary:
        word → [[morph, gloss], [morph2, gloss2], ...]

    - Prints unique warnings once with count at the end.
    - Keeps track of how many times each inconsistency appears.
    """

    if existing_dict is None:
        existing_dict = {}

    # dictionary: warning_text → count
    if warnings_counter is None:
        warnings_counter = defaultdict(int)

    # ----------------------------------
    # Parse file and find tiers
    # ----------------------------------
    tree = ET.parse(filename)
    root = tree.getroot()

    def find_tier(root, tier_name):
        for t in root.findall(".//TIER"):
            if t.get("TIER_ID") == tier_name:
                return t
        return None

    words_tier = find_tier(root, word_tiername)
    morph_tier = find_tier(root, morph_tiername)
    gloss_tier = find_tier(root, gloss_tiername)

    if (words_tier is None) or (morph_tier is None) or (gloss_tier is None):
        raise ValueError(
            f"Missing tiers in file {filename}:\n"
            f"  Words={words_tier is not None}, "
            f"  Morph={morph_tier is not None}, "
            f"  Gloss={gloss_tier is not None}"
        )

    # ----------------------------------
    # Helpers: build annotation maps
    # ----------------------------------
    def build_ref_map(tier_elem):
        m = {}
        for ann in tier_elem.findall("ANNOTATION"):
            ref = ann.find("REF_ANNOTATION")
            if ref is None:
                continue
            ann_id = ref.get("ANNOTATION_ID")
            val_el = ref.find("ANNOTATION_VALUE")
            text = (val_el.text or "").strip() if val_el is not None and val_el.text else ""
            if ann_id:
                m[ann_id] = text
        return m

    word_map = build_ref_map(words_tier)     # word_id → word text
    morph_map = build_ref_map(morph_tier)    # morph_id → morph text

    # word_id → list of morph_ids in order
    word_to_morph_ids = {}
    for ann in morph_tier.findall("ANNOTATION"):
        ref = ann.find("REF_ANNOTATION")
        if ref is None:
            continue
        morph_id = ref.get("ANNOTATION_ID")
        parent_word_id = ref.get("ANNOTATION_REF")
        if morph_id and parent_word_id:
            word_to_morph_ids.setdefault(parent_word_id, []).append(morph_id)

    # morph_id → gloss_text
    morph_to_gloss = {}
    for ann in gloss_tier.findall("ANNOTATION"):
        ref = ann.find("REF_ANNOTATION")
        if ref is None:
            continue
        morph_id = ref.get("ANNOTATION_REF")
        val_el = ref.find("ANNOTATION_VALUE")
        gloss_text = (val_el.text or "").strip() if val_el is not None and val_el.text else ""
        if morph_id:
            morph_to_gloss[morph_id] = gloss_text

    # ----------------------------------
    # Build combined dictionary
    # ----------------------------------
    added = 0
    skipped = 0

    for word_id, word_text in word_map.items():
        morph_ids = word_to_morph_ids.get(word_id, [])
        if not morph_ids:
            continue

        pair_seq = []
        for mid in morph_ids:
            morph_text = morph_map.get(mid, "")
            gloss_text = morph_to_gloss.get(mid, "")
            pair_seq.append([morph_text, gloss_text])

        if word_text in existing_dict:
            if existing_dict[word_text] != pair_seq:
                # generate warning message
                warning = (
                    f"⚠ Warning: '{word_text}' already mapped to "
                    f"{existing_dict[word_text]}; encountered new {pair_seq}. Keeping existing."
                )
                warnings_counter[warning] = warnings_counter.get(warning, 0) + 1
                skipped += 1
            else:
                skipped += 1  # identical case
        else:
            existing_dict[word_text] = pair_seq
            added += 1

    # ----------------------------------
    # Save dictionary
    # ----------------------------------
    with open(save_json_path, "w", encoding="utf-8") as f:
        json.dump(existing_dict, f, ensure_ascii=False, indent=2)

    print(
        f"{filename}: Added {added}, skipped {skipped}. "
        f"Current total entries: {len(existing_dict)}"
    )

    return existing_dict, warnings_counter

In [ ]:
def print_warning_summary(warnings_counter):
    print("\n=== UNIQUE WARNING SUMMARY ===")
    if not warnings_counter:
        print("No conflicts found. ✔️")
        return
    for txt, count in warnings_counter.items():
        print(f"{txt}  ({count}×)")

In [ ]:
def save_warnings_to_csv(warnings_counter, csv_path="warnings_summary.csv"):
    """
    Save unique warnings and their counts into a CSV file.

    Columns:
        warning_text,count
    """
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["warning_text", "count"])

        for warning, count in warnings_counter.items():
            writer.writerow([warning, count])

    print(f"Warnings saved to {csv_path}")

In [ ]:
# List of files and tier triplets
configs = [
    {
        "filename": "i013.eaf",
        "triplets": [
            ("Speaker1-words", "Speaker1-morph", "Speaker1-gloss")
        ],
    },
    {
        "filename": "i021.eaf",
        "triplets": [
            ("Speaker2-token", "Speaker2-morph", "Speaker2-gloss"),
            ("Speaker3-token",    "Speaker3-morph",    "Speaker3-gloss"),
            ("Speaker4-token",    "Speaker4-morph",    "Speaker4-gloss")
        ],
    }
]

In [ ]:
# Start from an empty dict (if rebuilding from scratch)
word_morph_gloss_dict = {}

In [ ]:
save_json_path = "word_morph_gloss_dict.json"

In [ ]:
warnings_counter = {}

In [ ]:
for cfg in configs:
    fname = cfg["filename"]
    for (w, m, g) in cfg["triplets"]:
        word_morph_gloss_dict, warnings_counter = extract_word_morph_gloss_dict(
            filename=fname,
            word_tiername=w,
            morph_tiername=m,
            gloss_tiername=g,
            existing_dict=word_morph_gloss_dict,
            warnings_counter=warnings_counter,
            save_json_path="word_morph_gloss_dict.json"
        )

In [ ]:
print_warning_summary(warnings_counter)

In [ ]:
save_warnings_to_csv(warnings_counter, "warnings_summary.csv")

# Corrections in the annotated files

In [ ]:
import pandas as pd
import ast
import xml.etree.ElementTree as ET
import os

In [ ]:
def load_corrections(csv_path="correct_in_files.csv"):
    """
    Load corrections from a CSV with columns:
      - token
      - if the encountered value is
      - correct to
      - language   (e.g. 'spa' / 'piem'; may be empty)

    Returns:
      corrections: dict keyed by
        (token, tuple(tuple(morph, gloss)...)) -> (new_seq, lang_tag)

      where:
        - new_seq is a list of [morph, gloss] pairs
        - lang_tag is a string (may be "")
    """
    df = pd.read_csv(csv_path)

    # Normalize column names just in case
    df.columns = [c.lower().strip() for c in df.columns]

    required = [
        "token",
        "if the encountered value is",
        "correct to",
        "language",
    ]
    if not all(col in df.columns for col in required):
        raise ValueError(f"CSV must contain columns: {required}. Found: {df.columns.tolist()}")

    corrections = {}

    for i, row in df.iterrows():
        token = str(row["token"]).strip()
        lang_tag = str(row["language"]).strip() if not pd.isna(row["language"]) else ""

        try:
            from_seq = ast.literal_eval(row["if the encountered value is"])
            to_seq   = ast.literal_eval(row["correct to"])
        except Exception as e:
            print(f"⚠ Skipping row {i} ('{token}'): parse error: {e}")
            continue

        # Key: (token, tuple of (morph, gloss) pairs)
        try:
            key = (token, tuple(tuple(p) for p in from_seq))
        except TypeError as e:
            print(f"⚠ Skipping row {i} ('{token}'): malformed from_seq: {e}")
            continue

        corrections[key] = (to_seq, lang_tag)

    print(f"Loaded {len(corrections)} correction rules from {csv_path}")
    return corrections

In [ ]:
def apply_corrections_to_file(filename, triplets, corrections):
    """
    Apply corrections to an ELAN .eaf file and save as <filename>_upd.eaf.

    Each triplet in `triplets` is:
      (word_tiername, morph_tiername, gloss_tiername, lang_tiername)

    For each word, we compare:
      current_seq = [[morph, gloss], ...]
    with keys in `corrections`, and if there's a match, apply:

      1) len(new_seq) == len(current_seq): overwrite morph+gloss
      2) len(new_seq) == 1 < len(current_seq): collapse many → one
          - delete extra morph+gloss + language annotations
      3) len(current_seq) == 1 < len(new_seq): expand one → many
          - add new morph+gloss annotations
          - add language annotations for new morphs with lang_tag
            (from the CSV "language" column)

    After all corrections, for each (morph, language) pair:
      - drop language annotations whose ANNOTATION_REF does not exist
        in the morph tier.

    Finally, updates HEADER/PROPERTY NAME="lastUsedAnnotationId"
    to the max annotation id used, and saves <filename>_upd.eaf.
    """
    print(f"\n=== Applying corrections in {filename} ===")

    tree = ET.parse(filename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}", 1)[0][1:]
    else:
        ns_uri = None

    def q(tag):
        """Qualify tag with namespace if present."""
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    # ----- Output filename -----
    base, ext = os.path.splitext(filename)
    outname = base + "_upd" + ext

    # ----- Find current max annotation id -----
    max_id_num = 0
    for ref in root.findall(f".//{q('REF_ANNOTATION')}"):
        aid = ref.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))
    for al in root.findall(f".//{q('ALIGNABLE_ANNOTATION')}"):
        aid = al.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))

    def new_ann_id():
        nonlocal max_id_num
        max_id_num += 1
        return f"a{max_id_num}"

    def find_tier(tier_name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == tier_name:
                return t
        return None

    total_token_corrections = 0

    # =================================================================
    # Process each (words, morph, gloss, lang) set
    # =================================================================
    for (word_tiername, morph_tiername, gloss_tiername, lang_tiername) in triplets:
        print(f"\n--- Triplet: {word_tiername} / {morph_tiername} / {gloss_tiername} / {lang_tiername} ---")

        words_tier = find_tier(word_tiername)
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)
        lang_tier  = find_tier(lang_tiername)

        if words_tier is None or morph_tier is None or gloss_tier is None:
            print(f"  ⚠ Skipping: one of the tiers not found.")
            continue

        if lang_tier is None:
            print(f"  ⚠ Language tier '{lang_tiername}' not found. Will only correct morph+gloss.")
        # -----------------------------
        # Build maps
        # -----------------------------

        # word_id → word_text
        word_texts = {}
        for ann in words_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            wid = ref.get("ANNOTATION_ID")
            val_el = ref.find(q("ANNOTATION_VALUE"))
            text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            if wid:
                word_texts[wid] = text

        # word_id → [morph_id1, morph_id2,...]
        word_to_morph_ids = {}
        # morph_id → (morph_text, morph_VALUE_el, morph_ANNOTATION_el)
        morph_info = {}
        for ann in morph_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            mid = ref.get("ANNOTATION_ID")
            parent = ref.get("ANNOTATION_REF")
            val_el = ref.find(q("ANNOTATION_VALUE"))
            text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            if mid:
                morph_info[mid] = (text, val_el, ann)
            if mid and parent:
                word_to_morph_ids.setdefault(parent, []).append(mid)

        # morph_id → (gloss_text, gloss_VALUE_el, gloss_ANNOTATION_el)
        gloss_info = {}
        for ann in gloss_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            mid = ref.get("ANNOTATION_REF")
            val_el = ref.find(q("ANNOTATION_VALUE"))
            text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            if mid:
                gloss_info[mid] = (text, val_el, ann)

        # morph_id → (lang_text, lang_VALUE_el, lang_ANNOTATION_el)
        lang_info = {}
        if lang_tier is not None:
            for ann in lang_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_REF")
                val_el = ref.find(q("ANNOTATION_VALUE"))
                text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
                if mid:
                    lang_info[mid] = (text, val_el, ann)

        # -----------------------------
        # Apply corrections token by token
        # -----------------------------
        for word_id, word_text in word_texts.items():
            morph_ids = word_to_morph_ids.get(word_id, [])
            if not morph_ids:
                continue

            # current [[morph, gloss], ...]
            current_seq = []
            for mid in morph_ids:
                m_text, _m_val, _m_ann = morph_info.get(mid, ("", None, None))
                g_text = gloss_info.get(mid, ("", None, None))[0]
                current_seq.append([m_text, g_text])

            key = (word_text, tuple(tuple(p) for p in current_seq))
            if key not in corrections:
                continue

            new_seq, lang_tag = corrections[key]
            lang_tag = lang_tag or ""  # normalize

            # ---------- Case 1: same length ----------
            if len(new_seq) == len(current_seq):
                print(f"  ✔ Updating '{word_text}': {current_seq} -> {new_seq}")
                for mid, (new_morph, new_gloss) in zip(morph_ids, new_seq):
                    # morph
                    if mid in morph_info:
                        _old, val_el, _ann = morph_info[mid]
                        if val_el is not None:
                            val_el.text = new_morph
                    # gloss
                    if mid in gloss_info:
                        _old_g, val_el_g, _ann_g = gloss_info[mid]
                        if val_el_g is not None:
                            val_el_g.text = new_gloss
                    # language: we leave as is (you'll still have piem/spa logic)
                total_token_corrections += 1
                continue

            # ---------- Case 2: collapse (many → one) ----------
            if len(new_seq) == 1 and len(current_seq) > 1:
                print(f"  ✔ Collapsing '{word_text}': {current_seq} -> {new_seq}")
                new_morph, new_gloss = new_seq[0]
                keep_id = morph_ids[0]

                # update first morph
                if keep_id in morph_info:
                    _old, val_el, _ann = morph_info[keep_id]
                    if val_el is not None:
                        val_el.text = new_morph
                # update first gloss
                if keep_id in gloss_info:
                    _old_g, val_el_g, _ann_g = gloss_info[keep_id]
                    if val_el_g is not None:
                        val_el_g.text = new_gloss
                # language for keep_id: leave as is if present

                # remove extra morph+gloss+language for mid in morph_ids[1:]
                for mid in morph_ids[1:]:
                    # morph
                    if mid in morph_info:
                        _t, _ve, ann_m = morph_info[mid]
                        if ann_m is not None and ann_m in list(morph_tier):
                            morph_tier.remove(ann_m)
                    # gloss
                    if mid in gloss_info:
                        _t, _ve, ann_g = gloss_info[mid]
                        if ann_g is not None and ann_g in list(gloss_tier):
                            gloss_tier.remove(ann_g)
                    # language
                    if lang_tier is not None and mid in lang_info:
                        _t, _ve, ann_l = lang_info[mid]
                        if ann_l is not None and ann_l in list(lang_tier):
                            lang_tier.remove(ann_l)

                total_token_corrections += 1
                continue

            # ---------- Case 3: expand (one → many) ----------
            if len(current_seq) == 1 and len(new_seq) > 1:
                print(f"  ✔ Expanding '{word_text}': {current_seq} -> {new_seq}")

                keep_id = morph_ids[0]
                keep_m_text, keep_val_el, keep_ann = morph_info.get(keep_id, ("", None, None))

                first_morph, first_gloss = new_seq[0]

                # update first morph/gloss
                if keep_val_el is not None:
                    keep_val_el.text = first_morph
                if keep_id in gloss_info:
                    _old_g, val_el_g, _ann_g = gloss_info[keep_id]
                    if val_el_g is not None:
                        val_el_g.text = first_gloss

                # update/create language for the first segment if lang_tag present
                if lang_tier is not None and lang_tag:
                    if keep_id in lang_info:
                        _old_l, val_el_l, _ann_l = lang_info[keep_id]
                        if val_el_l is not None:
                            val_el_l.text = lang_tag
                    else:
                        # create new language annotation
                        lang_ann = ET.SubElement(lang_tier, q("ANNOTATION"))
                        lang_ref = ET.SubElement(lang_ann, q("REF_ANNOTATION"), {
                            "ANNOTATION_ID": new_ann_id(),
                            "ANNOTATION_REF": keep_id
                        })
                        lang_val = ET.SubElement(lang_ref, q("ANNOTATION_VALUE"))
                        lang_val.text = lang_tag

                # Insert extra morph+gloss (+language) after keep_ann in the morph tier
                morph_children = list(morph_tier.findall(q("ANNOTATION")))
                try:
                    base_idx = morph_children.index(keep_ann)
                except ValueError:
                    base_idx = len(morph_children) - 1

                prev_mid = keep_id

                for offset, (morph_text, gloss_text) in enumerate(new_seq[1:], start=1):
                    # new morph
                    new_mid = new_ann_id()
                    morph_ann = ET.Element(q("ANNOTATION"))
                    ref = ET.SubElement(morph_ann, q("REF_ANNOTATION"), {
                        "ANNOTATION_ID": new_mid,
                        "ANNOTATION_REF": word_id,
                        "PREVIOUS_ANNOTATION": prev_mid
                    })
                    val_el = ET.SubElement(ref, q("ANNOTATION_VALUE"))
                    val_el.text = morph_text
                    morph_tier.insert(base_idx + offset, morph_ann)

                    # new gloss
                    gloss_ann = ET.SubElement(gloss_tier, q("ANNOTATION"))
                    gloss_ref = ET.SubElement(gloss_ann, q("REF_ANNOTATION"), {
                        "ANNOTATION_ID": new_ann_id(),
                        "ANNOTATION_REF": new_mid
                    })
                    gloss_val = ET.SubElement(gloss_ref, q("ANNOTATION_VALUE"))
                    gloss_val.text = gloss_text

                    # new language
                    if lang_tier is not None and lang_tag:
                        lang_ann = ET.SubElement(lang_tier, q("ANNOTATION"))
                        lang_ref = ET.SubElement(lang_ann, q("REF_ANNOTATION"), {
                            "ANNOTATION_ID": new_ann_id(),
                            "ANNOTATION_REF": new_mid
                        })
                        lang_val = ET.SubElement(lang_ref, q("ANNOTATION_VALUE"))
                        lang_val.text = lang_tag

                    prev_mid = new_mid

                total_token_corrections += 1
                continue

            # ---------- Other mismatch ----------
            print(
                f"  ⚠ Length mismatch for '{word_text}': "
                f"existing {current_seq} vs correction {new_seq}. Skipping."
            )

        # -----------------------------
        # Cleanup: remove invalid language refs
        # -----------------------------
        if lang_tier is not None:
            # recompute current morph IDs in this tier
            current_morph_ids = set()
            for ann in morph_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_ID")
                if mid:
                    current_morph_ids.add(mid)

            to_remove = []
            for ann in lang_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_REF")
                if mid and mid not in current_morph_ids:
                    to_remove.append((ann, ref.get("ANNOTATION_ID"), mid))

            if to_remove:
                print(f"  ✂ Cleaning {len(to_remove)} invalid language annotations in '{lang_tiername}'")
                for ann, aid, mid in to_remove:
                    lang_tier.remove(ann)
                    # print(f"    removed language ann {aid} (ref {mid})")

    # =================================================================
    # Update lastUsedAnnotationId in HEADER
    # =================================================================
    header = root.find(q("HEADER"))
    if header is not None:
        prop_last = None
        for prop in header.findall(q("PROPERTY")):
            if prop.get("NAME") == "lastUsedAnnotationId":
                prop_last = prop
                break
        if prop_last is None:
            prop_last = ET.SubElement(header, q("PROPERTY"), {"NAME": "lastUsedAnnotationId"})
        prop_last.text = str(max_id_num)

    # =================================================================
    # Save as new file
    # =================================================================
    tree.write(outname, encoding="utf-8", xml_declaration=True)
    print(f"\n=== Saved corrected file as: {outname} ===")
    print(f"    Total corrected tokens: {total_token_corrections}")

In [ ]:
corrections = load_corrections("correct_in_files.csv")

In [ ]:
print(corrections)

In [ ]:
# List of files and tier triplets
configs = [
    {
        "filename": "new template file (pilar-i013).eaf",
        "triplets": [
            ("Speaker1-words", "Speaker1-morph", "Speaker1-gloss", "Speaker1-language")
        ],
    },
    {
        "filename": "pilar-i021_upd.eaf",
        "triplets": [
            ("Speaker2-token", "Speaker2-morph", "Speaker2-gloss", "Speaker2-language"),
            ("Speaker3-token",    "Speaker3-morph",    "Speaker3-gloss", "Speaker3-language"),
            ("Speaker4-token",    "Speaker4-morph",    "Speaker4-gloss", "Speaker4-language")
        ],
    }
]

In [ ]:
for cfg in configs:
    fname = cfg["filename"]
    triplets = cfg["triplets"]
    apply_corrections_to_file(fname, triplets, corrections)

print("✅ All corrections applied to ELAN files.")

In [ ]:
filename = "new template file (pilar-i013)_upd.eaf"          # your uploaded file
morph_tier_id = "Speaker1-morph"
output = "i013_clean.eaf"      # new file to write

tree = ET.parse(filename)
root = tree.getroot()

def find_tier(root, tid):
    for t in root.findall(".//TIER"):
        if t.get("TIER_ID") == tid:
            return t
    return None

morph_tier = find_tier(root, morph_tier_id)
if morph_tier is None:
    raise ValueError(f"Morph tier '{morph_tier_id}' not found")

# Collect existing morph IDs
morph_ids = set()
for ann in morph_tier.findall("ANNOTATION"):
    ref = ann.find("REF_ANNOTATION")
    if ref is None:
        continue
    mid = ref.get("ANNOTATION_ID")
    if mid:
        morph_ids.add(mid)

print("Morph IDs:", len(morph_ids))

# Find all child tiers of the morph tier (e.g. Speaker1-gloss, Speaker1-language)
child_tiers = []
for t in root.findall(".//TIER"):
    if t.get("PARENT_REF") == morph_tier_id:
        child_tiers.append(t)

print("Child tiers of", morph_tier_id, ":", [t.get("TIER_ID") for t in child_tiers])

# For each child tier, remove annotations that point to non-existent morph IDs
for child in child_tiers:
    tid = child.get("TIER_ID")
    to_remove = []
    for ann in child.findall("ANNOTATION"):
        ref = ann.find("REF_ANNOTATION")
        if ref is None:
            continue
        mid = ref.get("ANNOTATION_REF")
        if mid and mid not in morph_ids:
            to_remove.append((ann, ref.get("ANNOTATION_ID"), mid))
    print(f"TIER {tid}: will remove {len(to_remove)} invalid annotations")
    for ann, aid, mid in to_remove:
        child.remove(ann)
        print(f"  removed language/gloss ann {aid} (ref {mid})")

# Save to new file
tree.write(output, encoding="utf-8", xml_declaration=True)
print("✅ Clean file written to", output)

# Add corrections to the dict

In [ ]:
import json
import pandas as pd
import ast

In [ ]:
dict_path = "word_morph_gloss_dict.json"
csv_path  = "correct_in_dict.csv"
output_path = "word_morph_gloss_dict_upd.json"

In [ ]:
with open(dict_path, "r", encoding="utf-8") as f:
    word_dict = json.load(f)

print(f"Loaded dictionary with {len(word_dict)} entries.")

In [ ]:
df = pd.read_csv(csv_path)

In [ ]:
num_updated = 0
num_added = 0

for idx, row in df.iterrows():
    token = str(row["token"]).strip()
    raw_value = row["correct to"]

    # Parse list-like string safely
    try:
        new_value = ast.literal_eval(raw_value)
    except Exception as e:
        print(f"⚠ Skipping row {idx}: cannot parse value '{raw_value}'. Error: {e}")
        continue

    if token in word_dict:
        print(f"✏️ Updating '{token}':")
        print(f"    old → {word_dict[token]}")
        print(f"    new → {new_value}")
        word_dict[token] = new_value
        num_updated += 1
    else:
        print(f"➕ Adding new token '{token}' → {new_value}")
        word_dict[token] = new_value
        num_added += 1

In [ ]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(word_dict, f, ensure_ascii=False, indent=2)

print("\n===== SUMMARY =====")
print(f"Updated entries: {num_updated}")
print(f"New entries added: {num_added}")
print(f"Total entries now: {len(word_dict)}")
print(f"Saved updated dictionary to: {output_path}")